# 2. Quantise the detector for the ESP32-S3

The ESP32-S3 runs int8 (and some int16) arithmetic through Espressif's
[ESP-DL](https://github.com/espressif/esp-dl). This notebook converts the trained PyTorch
checkpoint into an `.espdl` file, step by step, and checks at each step that nothing is lost.
`scripts/esp32_export.py` does the same in one command.

1. Export the network to ONNX **without** the box decoding: raw per-stride box distances and class logits.
2. Decode those raw outputs in Python exactly as the firmware will, and check against ultralytics.
3. Calibrate and quantise with [esp-ppq](https://github.com/espressif/esp-ppq).
4. Compare float vs quantised: mAP **and recall at the device's 0.25 threshold**.
5. Look at per-image scores (where plain int8 goes wrong) and at the boxes.
6. Copy the model into the firmware and flash it.

**Needs:** `pip install -r requirements-esp32.txt` (Linux or macOS, Python 3.10), the dataset (the
train split is the calibration data), and a CUDA GPU for step 3's blockwise reconstruction
(about 30 min on an H100; without it the int8 model misses most fish, see step 4).

In [ ]:
import os, sys
from pathlib import Path

# The notebooks live in notebooks/; the shared code is the rabbitfish package at the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from rabbitfish.paths import DATASET, SAMPLES, RUNS, MODELS, FIRMWARE, WEIGHTS  # also sets YOLO_CONFIG_DIR

# Quick mode (RABBITFISH_QUICK=1) shrinks every expensive step so the whole notebook runs in a few
# minutes. It is how the notebooks are tested; the numbers it produces are meaningless.
QUICK = os.environ.get("RABBITFISH_QUICK") == "1"
print("repository:", ROOT)
print("dataset:   ", DATASET, "(found)" if (DATASET / "data.yaml").exists() else "(MISSING: see data/README.md)")
print("quick mode:", QUICK)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("quantisation device:", DEVICE)

In [ ]:
# Which checkpoint to quantise: the committed one, or your own run from notebook 01.
WEIGHTS_PATH = MODELS / "yolo26n-i320" / "best.pt"
# WEIGHTS_PATH = RUNS / "yolo26n-b8-i320" / "weights" / "best.pt"

WIDTH, HEIGHT = 320, 256   # model input; frames are 4:3 and both sides must be multiples of 32
HEAD = "o2m"               # YOLO26's one-to-many head (needs NMS); the NMS-free o2o head quantises badly
CLS_CLAMP = 10             # clamp class logits to +-10 inside the model (see step 1)
BLOCKWISE = DEVICE == "cuda"   # see step 3: needs a GPU; without it, expect the warning in step 4
CALIB_IMAGES = 16 if QUICK else 256
EVAL_LIMIT = 8 if QUICK else None       # images used for the scores in step 4 (None: the whole val split)

OUT = RUNS / "notebook-export" / (f"{WIDTH}x{HEIGHT}-w8a8-{HEAD}-c{CLS_CLAMP}-box16" + ("-br" if BLOCKWISE else ""))
OUT.mkdir(parents=True, exist_ok=True)
print("output folder:", OUT)

## Step 1: ONNX export of the raw head

A YOLO26 detection head ends in, per stride 8/16/32:

- `box_s<stride>` (1, 4, h, w): distances from the cell centre to the left/top/right/bottom box edges, in cells
- `cls_s<stride>` (1, 2, h, w): class logits for fishbody and fishhead

Ultralytics normally decodes these inside the model (anchors, concatenation, top-k). We stop
before that, because (a) boxes and logits have very different ranges and quantise much better as
separate tensors with their own scales, and (b) the device then needs no TopK/Gather operators.
The decoding is a few lines of C++ in the firmware instead.

`CLS_CLAMP`: on certain background the logits reach about -170. Clamping to +-10 inside the model
(sigmoid(-10) = 0.00005, i.e. the same decision) stops those values from stretching the int8 range.

In [ ]:
from rabbitfish import export

onnx_path = OUT / "model.onnx"
strides, out_names, names = export.export_onnx(WEIGHTS_PATH, onnx_path, WIDTH, HEIGHT, HEAD, CLS_CLAMP)
export.simplify_onnx(onnx_path)
print("strides:", strides)
print("outputs:", out_names)
print("classes:", names)

## Step 2: preprocessing and decoding, exactly as on the device

`letterbox` scales the frame into 320x256 keeping the aspect ratio, pads with grey (114) and
divides by 255. `decode` turns the raw maps into boxes and scores and applies per-class NMS.
The firmware's `Detector::preprocess` and `Detector::postprocess` are line-by-line ports of
these two functions, so whatever is measured here is what the device computes (up to the int8
rounding we add in step 3).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

COLORS = {0: (232, 89, 12), 1: (25, 113, 194)}  # fishbody orange, fishhead blue (RGB), as on the device's page
NAMES = {0: "fishbody", 1: "fishhead"}

def show(img_bgr, boxes=(), classes=(), scores=None, title="", ax=None):
    """Draw xyxy boxes (in img_bgr's pixels) on an image."""
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).copy()
    lw = max(2, img.shape[1] // 400)
    for i, (b, c) in enumerate(zip(boxes, classes)):
        x1, y1, x2, y2 = map(int, b)
        cv2.rectangle(img, (x1, y1), (x2, y2), COLORS[int(c)], lw)
        label = NAMES[int(c)] + (f" {scores[i]:.2f}" if scores is not None else "")
        cv2.putText(img, label, (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, lw / 3, COLORS[int(c)], lw)
    ax = ax or plt.figure(figsize=(8, 6)).gca()
    ax.imshow(img); ax.set_title(title); ax.axis("off")

import onnxruntime as ort
from rabbitfish.data import images, letterbox, load_labels
from rabbitfish.metrics import decode

so = ort.SessionOptions()
so.intra_op_num_threads = 4   # explicit, or onnxruntime tries to pin threads to cores (fails in containers/clusters)
sess = ort.InferenceSession(str(onnx_path), so, providers=["CPUExecutionProvider"])
float_fn = lambda x: sess.run(out_names, {"images": x})

sample = images("samples", root=SAMPLES.parent)[0]
img = cv2.imread(str(sample))
x, (r, px, py) = letterbox(img, WIDTH, HEIGHT)
outs = float_fn(x)
for n, o in zip(out_names, outs):
    print(f"{n:8s} {tuple(o.shape)}  range {o.min():8.2f} .. {o.max():6.2f}")

boxes, scores, cls = decode(outs, strides, conf_thr=0.25, nms_iou=0.7)
boxes = (boxes - [px, py, px, py]) / r   # model-input pixels -> original image pixels
show(img, boxes, cls, scores, title=f"float ONNX on {sample.name}")

## Step 3: calibrate and quantise with esp-ppq

esp-ppq runs the model on calibration images (letterboxed train images) to choose a power-of-two
scale for every tensor, then writes the `.espdl` file ESP-DL loads.

Settings, and why:

- **w8a8**: int8 weights and activations. 16-bit activations (w8a16) would be more accurate in
  simulation, but ESP-DL 3.3 on the device does not reproduce esp-ppq's int16 arithmetic (its
  self-test fails), and it is 50% slower.
- **int16 for the three final box convolutions** (`box16`): box distances need finer steps than
  int8 gives; this is cheap and the device reproduces it exactly.
- **Blockwise reconstruction** (`BLOCKWISE`): esp-ppq fine-tunes the rounding of each block's
  weights to reproduce the float block outputs on the calibration images. **Without it, this
  model's class scores collapse after int8 quantisation** (fish that score 0.8 in float score
  0.01-0.3), so at the usual 0.25 threshold the device misses about half the fish, even though
  mAP only drops from 0.96 to 0.92. More int16 layers do not fix this; blockwise reconstruction
  does (`results/esp32/quantisation_study.md`). It needs a GPU (~30 min on an H100).

The `.espdl` also stores one val image and the outputs esp-ppq computed for it. At boot the
firmware runs that image and compares bit for bit: the "self-test" shown on the device's page.

In [ ]:
import time

int16_ops = export.node_names(onnx_path, producing=[n for n in out_names if n.startswith("box")])
print("int16 ops:", int16_ops)

t0 = time.time()
graph = export.quantise(onnx_path, OUT / "model.espdl", out_names, WIDTH, HEIGHT, quant="w8a8",
                        int16_ops=int16_ops, blockwise=BLOCKWISE, blockwise_steps=20 if QUICK else None,
                        calib_images=CALIB_IMAGES, device=DEVICE)
print(f"quantised in {time.time() - t0:.0f} s -> {OUT / 'model.espdl'} "
      f"({(OUT / 'model.espdl').stat().st_size / 1e6:.1f} MB)")

## Step 4: float vs quantised on the val split

Both models go through the same `evaluate`: letterbox, run, decode, ultralytics' matching and
101-point AP. Besides mAP it reports precision and **recall at conf 0.25**, the device's default
threshold. That is the number to watch: mAP only depends on how predictions are ranked, so it
stays high even when every score is squashed towards zero, but R@0.25 is what a user sees.

In [ ]:
import pandas as pd
from rabbitfish import metrics

_, quant_fn = export.runners(onnx_path, graph, out_names, DEVICE)
nms_iou = 0.7 if HEAD == "o2m" else None
res = {
    "float (ONNX)": metrics.evaluate(float_fn, strides, WIDTH, HEIGHT, names, "val", nms_iou, EVAL_LIMIT),
    "int8 (esp-ppq simulated)": metrics.evaluate(quant_fn, strides, WIDTH, HEIGHT, names, "val", nms_iou, EVAL_LIMIT),
}
keys = ["mAP50", "mAP50-95", "P@0.25", "R@0.25"]
table = pd.DataFrame({k: {m: round(v[m], 3) for m in keys} for k, v in res.items()}).T
if table.loc["int8 (esp-ppq simulated)", "R@0.25"] < 0.9 * table.loc["float (ONNX)", "R@0.25"]:
    print("WARNING: the int8 model loses recall at 0.25: its scores are squashed. Use BLOCKWISE = True (GPU).")
table

## Step 5: per-image scores and boxes

For a few samples: the best score on each labelled object, float vs int8, and the int8 boxes.
With blockwise reconstruction the int8 scores are within a few percent of the float ones.

In [ ]:
from rabbitfish.metrics import match

rows = []
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, p in zip(axes.ravel(), images("samples", root=SAMPLES.parent)[:4]):
    img = cv2.imread(str(p))
    x, sp = letterbox(img, WIDTH, HEIGHT)
    tcls, tboxes = load_labels(p, img.shape, sp)
    for model_name, fn in [("float", float_fn), ("int8", quant_fn)]:
        b, s, c = decode(fn(x), strides, 0.001, nms_iou=0.7)
        tp = match(c, b, tcls, tboxes)[:, 0]     # true positives at IoU 0.5
        best = [float(s[(c == tc) & tp].max()) if ((c == tc) & tp).any() else 0.0 for tc in sorted(set(tcls))]
        rows.append({"image": p.name, "model": model_name,
                     **{names[int(tc)]: round(v, 2) for tc, v in zip(sorted(set(tcls)), best)}})
        if model_name == "int8":
            keep = s >= 0.25
            r, px, py = sp
            show(img, (b[keep] - [px, py, px, py]) / r, c[keep], s[keep], title=f"int8: {p.name}", ax=ax)
plt.tight_layout()
pd.DataFrame(rows)

## Step 6: put the model on the device

`select_model.py` copies `model.espdl` into `firmware/rabbitfish_server/model/` and writes
`model_info.json` (classes, strides, input size, the mAP shown on the device's page) from a
`report.json`. We write that report here, then flash; see `firmware/rabbitfish_server/README.md`
for the ESP-IDF setup.

In [ ]:
import json, subprocess

report = {
    "run": WEIGHTS_PATH.parent.parent.name if WEIGHTS_PATH.parent.name == "weights" else WEIGHTS_PATH.parent.name,
    "head": HEAD, "quant": "w8a8", "cls_clamp": CLS_CLAMP, "blockwise": BLOCKWISE, "int16_ops": int16_ops,
    "test_image": images("val", 1)[0].name, "strides": strides, "classes": names, "outputs": out_names,
    "input": {"width": WIDTH, "height": HEIGHT, "layout": "NCHW RGB", "scale": "1/255", "letterbox_pad": 114},
    "mAP_float": res["float (ONNX)"], "mAP_quant": res["int8 (esp-ppq simulated)"], "eval_limit": EVAL_LIMIT,
}
(OUT / "report.json").write_text(json.dumps(report, indent=2))
print(f"wrote {OUT / 'report.json'}")
print("to install it in the firmware, run from the repository root:")
print(f"  python firmware/rabbitfish_server/tools/select_model.py {OUT.relative_to(ROOT)}")
print("then, in firmware/rabbitfish_server with ESP-IDF active:  idf.py -p <port> flash monitor")